# Historian Events Loader

This notebook processes the historian and events data for a given alarm tag.

**Workflow:**
1. Load the alarm tag's related-tag information from the Excel workbook (`UC3_Alarm_Tags_Presentation_30MAY2026.xlsx`).
2. Extract DCS tag names and their associated asset IDs from the sheet.
3. Map those short asset IDs (e.g. `1E`, `1F`) to UUIDs using the `ADNOC-B.json` config file.
4. Discover which UUID folders exist under `Historian_Events/Historian/` and `Historian_Events/events/`.
5. Load the historian time-series parquet files for each related tag.
6. Load the event parquet files (_E, _EA, _EAL) for each asset.

**Target tag for this run:** `03TIC_1023` (sheet: `03TIC_1023 PVLO_PVHI`)

In [4]:
import pandas as pd

df = pd.read_parquet('/home/h604827/ControlActions/DATA/combined_events/03PI_1655_PVHI_combined_events.parquet')
df

,Action,Actor,AreaName,AlarmLimit,Block,Category,ConditionName,Description,EventID,Flags,...,TagName,Area,Limit,EventTypeID,VT_End,LastEvent,NextEvent,rowid,Seconds,PrevSeconds
0,OK,None,1O,50.0,None,12.0,PVLO,3E112 O/L TEMP CNTRL,30951607,512.0,...,None,None,NaN,NaN,None,NaN,NaN,NaN,NaN,NaN
1,None,None,1O,50.0,None,12.0,PVLO,3E112 O/L TEMP CNTRL,30952437,512.0,...,None,None,NaN,NaN,None,NaN,NaN,NaN,NaN,NaN
2,OK,None,1O,50.0,None,12.0,PVLO,3E112 O/L TEMP CNTRL,30952673,512.0,...,None,None,NaN,NaN,None,NaN,NaN,NaN,NaN,NaN
3,None,None,1O,50.0,None,12.0,PVLO,3E112 O/L TEMP CNTRL,30953571,512.0,...,None,None,NaN,NaN,None,NaN,NaN,NaN,NaN,NaN
4,OK,None,1O,50.0,None,12.0,PVLO,3E112 O/L TEMP CNTRL,30953908,512.0,...,None,None,NaN,NaN,None,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
708819,None,None,None,NaN,None,NaN,None,3F102 ACCUMULATOR LEVEL,79474394,NaN,...,03LI_1654,1O,15.0,1.0,None,2.0,0.0,16475.0,6.0,761219.0
708820,ACK,None,1O,NaN,None,7.0,PVLO,PVLO HIGH 3F102 ACCUMULA...,79474395,NaN,...,None,None,NaN,NaN,None,NaN,NaN,NaN,NaN,NaN
708821,None,None,None,NaN,None,NaN,None,PVLO HIGH 3F102 ACCUMULA...,79474395,NaN,...,03LI_1654,1O,NaN,4.0,None,NaN,NaN,NaN,NaN,NaN
708822,OK,None,1O,60.0,None,12.0,PVHI,3E112 O/L TEMP CNTRL,79474396,512.0,...,None,None,NaN,NaN,None,NaN,NaN,NaN,NaN,NaN


## 0. Configuration

In [5]:
# ── Paths ──────────────────────────────────────────────────────────────────────
import os

REPO_ROOT          = os.path.abspath(os.path.join(os.path.dirname(os.getcwd()), '.'))
DATA_DIR           = os.path.join(REPO_ROOT, 'DATA')
HISTORIAN_ROOT     = os.path.join(DATA_DIR, 'Historian_Events')
HISTORIAN_DIR      = os.path.join(HISTORIAN_ROOT, 'Historian')   # tag time-series
EVENTS_DIR         = os.path.join(HISTORIAN_ROOT, 'events')       # alarm/event tables
EXCEL_FILE         = os.path.join(DATA_DIR, 'UC3_Alarm_Tags_Presentation_30MAY2026.xlsx')
ADNOC_JSON         = os.path.join(DATA_DIR, 'config', 'config', 'EMDB', 'ADNOC-B.json')

# ── Target alarm tag ──────────────────────────────────────────────────────────
TARGET_TAG         = '03PIC1023'
EXCEL_HEADER_ROW   = 6      # 0-indexed row that contains column headers

# ── Per-tag asset ID overrides ─────────────────────────────────────────────────
# If a tag's assets are known and should not be auto-derived from the Excel sheet,
# list them explicitly here. Keys are TARGET_TAG values.
ASSET_ID_OVERRIDE = {
    '03LIC_1071': ['1E', '1F', '1I', '1K', '1L'],
}

# Auto-detect the sheet whose name contains TARGET_TAG
import openpyxl
_wb = openpyxl.load_workbook(EXCEL_FILE, read_only=True, data_only=True)
_matches = [s for s in _wb.sheetnames if TARGET_TAG in s]
_wb.close()
if not _matches:
    raise ValueError(f"No sheet found containing '{TARGET_TAG}' in {EXCEL_FILE}")
SHEET_NAME = _matches[0]


print(f"REPO_ROOT      : {REPO_ROOT}")
print(f"SHEET_NAME     : {SHEET_NAME}  (auto-detected)")

print(f"EXCEL_FILE     : {EXCEL_FILE}")
print(f"TARGET_TAG     : {TARGET_TAG}")

print(f"ADNOC_JSON     : {ADNOC_JSON}")
print(f"EVENTS_DIR     : {EVENTS_DIR}")
print(f"HISTORIAN_DIR  : {HISTORIAN_DIR}")

REPO_ROOT      : /home/h604827/ControlActions
SHEET_NAME     : 03PIC1023 PVLO_PVHI  (auto-detected)
EXCEL_FILE     : /home/h604827/ControlActions/DATA/UC3_Alarm_Tags_Presentation_30MAY2026.xlsx
TARGET_TAG     : 03PIC1023
ADNOC_JSON     : /home/h604827/ControlActions/DATA/config/config/EMDB/ADNOC-B.json
EVENTS_DIR     : /home/h604827/ControlActions/DATA/Historian_Events/events
HISTORIAN_DIR  : /home/h604827/ControlActions/DATA/Historian_Events/Historian


## 1. Imports

In [6]:
import json
import pandas as pd

print(f"pandas  : {pd.__version__}")

pandas  : 2.3.3


## 2. Load Related-Tag Information from Excel

Each sheet in the Excel workbook describes the related tags for one alarm.  
We extract **DCS Tag Name**, **DESCRIPTION**, **PARAMETER**, **IS DATA AVAILABLE?** and **ASSET ID**.

In [7]:
print(f"Reading sheet '{SHEET_NAME}' from:\n  {EXCEL_FILE}\n")

df_sheet = pd.read_excel(EXCEL_FILE, sheet_name=SHEET_NAME, header=EXCEL_HEADER_ROW)

print(f"Raw sheet shape  : {df_sheet.shape}")
print(f"Columns ({len(df_sheet.columns)}):")
for c in df_sheet.columns:
    print(f"  {c}")

Reading sheet '03PIC1023 PVLO_PVHI' from:
  /home/h604827/ControlActions/DATA/UC3_Alarm_Tags_Presentation_30MAY2026.xlsx

Raw sheet shape  : (35, 32)
Columns (32):
  Remarks
  Potential Input parameter
  Description
  Causes (PVLO)
  Causes (PVHI)
  Plant
  P&ID
  SME Review (PV Low)
  SME Review (PV High)
  Unnamed: 9
  Unnamed: 10
  Unnamed: 11
  Unnamed: 12
  MH Comments
  DS Comments
  DCS Tag Name
  DESCRIPTION
  PARAMETER
  IS DATA AVAILABLE?
  SP-HIGH LIMIT
  SP-LOW LIMIT
  EXTENDED PV HIGH LIMIT
  PV-HIGH LIMIT
  EXTENDED PV LOW LIMIT
  PV-LOW LIMIT
  OP-HIGH LIMIT
  OP-LOW LIMIT
  PHD_TAG AVILABLE
  FILE NAME
  ASSET ID
  AVAILABLE IN APC
  SAFETY MANAGER TAGS


In [8]:
# Keep only rows that have a DCS Tag Name
KEEP_COLS = ['Potential Input parameter', 'DCS Tag Name', 'DESCRIPTION', 'PARAMETER',
             'IS DATA AVAILABLE?', 'ASSET ID']

df_tags = (
    df_sheet[KEEP_COLS]
    .dropna(subset=['DCS Tag Name'])
    .copy()
)

# Normalise strings
df_tags['DCS Tag Name'] = df_tags['DCS Tag Name'].str.strip()
df_tags['ASSET ID']     = df_tags['ASSET ID'].astype(str).str.strip()

# Reset index for clean display
df_tags = df_tags.reset_index(drop=True)

print(f"Related tags found: {len(df_tags)}")
print()
display(df_tags)

Related tags found: 34



,Potential Input parameter,DCS Tag Name,DESCRIPTION,PARAMETER,IS DATA AVAILABLE?,ASSET ID
0,03LICA_1016,03LIC_1016,3E104 LEVEL,PV,YES,1E
1,03TI_1405,03TI_1405,3E103 TUBE O/L TEMP,PV,YES,1E
2,03PCV_1023A,03HIC_1023A,SPLIT RANGE 03PIC_1023,OP,YES,1E
3,03PCV_1023B,03HIC_1023B,SPLITE RANGE 03PIC_1023,OP,YES,1E
4,03TIC_1023,03TIC_1023,3C101 OVHD TO 3V101 TEMP,PV,YES,1E
5,03TI_1015,03TI_1015,C3 I/L TO 3E104 TEMP T/C,PV,YES,1E
6,03PDIA_1020,03PDI_1020,DP ACROSS 3E104,PV,YES,1E
7,03TI_1404,03TI_1404,3E102 TUBE O/L TEMP,PV,YES,1E
8,03TI_1005,03TI_1005,3E102 TUBE I/L TEMP,PV,YES,1E
9,03PIC_1023,03PIC_1023,C3 3E104 TO 3C107 PRES,PV,YES,1E


## 3. Load ADNOC-B.json and Build Asset-ID → UUID Mapping

The JSON config contains one entry per asset node in the ADNOC Buhasa plant hierarchy.  
Leaf nodes whose `Name` matches the short asset ID in the sheet provide the UUID we need.

In [9]:
print(f"Reading asset config:\n  {ADNOC_JSON}\n")

with open(ADNOC_JSON, 'r') as f:
    asset_nodes = json.load(f)

print(f"Total nodes in ADNOC-B.json : {len(asset_nodes)}")

# Build a flat map:  short-name (e.g. '1E') → UUID
# We only care about leaf nodes under TRAIN_1 (adjust if other areas are needed)
asset_id_to_uuid = {}
for node in asset_nodes:
    asset_id_to_uuid[node['Name']] = {
        'uuid'      : node['Id'],
        'hierarchy' : node['Hierarchy'],
        'leaf'      : node['LeafNode'],
    }

print(f"Unique asset names mapped    : {len(asset_id_to_uuid)}")
print()

# Show all leaf nodes for easy inspection
leaf_map = pd.DataFrame(
    [{'Asset Name': k, 'UUID': v['uuid'], 'Hierarchy': v['hierarchy'], 'IsLeaf': v['leaf']}
     for k, v in asset_id_to_uuid.items()]
).sort_values('Asset Name').reset_index(drop=True)

print("Full asset map (all nodes):")
display(leaf_map)

Reading asset config:
  /home/h604827/ControlActions/DATA/config/config/EMDB/ADNOC-B.json

Total nodes in ADNOC-B.json : 74
Unique asset names mapped    : 74

Full asset map (all nodes):


,Asset Name,UUID,Hierarchy,IsLeaf
0,1A,8b999a4a-2a1e-4d6c-b76d-7a498fb5c4a3,ADNOC\BUHASA\ADNOC-B\TRAIN_1\1A,True
1,1B,f0c70dd0-139e-4102-b75d-77bccee3ca83,ADNOC\BUHASA\ADNOC-B\TRAIN_1\1B,True
2,1C,c71817e8-5df3-4ad6-89fb-0b71ec7d9f96,ADNOC\BUHASA\ADNOC-B\TRAIN_1\1C,True
3,1D,9cac06f4-131b-4af0-aea0-17b223973d12,ADNOC\BUHASA\ADNOC-B\TRAIN_1\1D,True
4,1E,072404dc-ae93-4239-9f6b-f4624391c391,ADNOC\BUHASA\ADNOC-B\TRAIN_1\1E,True
...,...,...,...,...
69,FIRE & GAS,de29a368-6e72-48a4-bbd2-84c32d2ac510,ADNOC\BUHASA\ADNOC-B\FIRE & GAS,False
70,LEAN_GAS,a663c56b-aafd-4a6a-837f-c8677655e9ca,ADNOC\BUHASA\ADNOC-B\LEAN_GAS,False
71,TRAIN_1,cdfe1dc7-d3db-4efe-a9b6-c5dec2902ccb,ADNOC\BUHASA\ADNOC-B\TRAIN_1,False
72,TRAIN_2,cc6268d1-0fe2-4fc2-8d84-79f6080b172d,ADNOC\BUHASA\ADNOC-B\TRAIN_2,False


## 4. Map Sheet Asset IDs → UUIDs

For every unique asset ID referenced in the sheet, look up the corresponding UUID.

In [10]:
if TARGET_TAG in ASSET_ID_OVERRIDE:
    unique_assets = ASSET_ID_OVERRIDE[TARGET_TAG]
    print(f"NOTE: Using hardcoded asset override for {TARGET_TAG}: {unique_assets}")
else:
    unique_assets = sorted(df_tags['ASSET ID'].dropna().unique())
print(f"Unique ASSET IDs in sheet : {unique_assets}")
print()

mapping_rows = []
for asset_name in unique_assets:
    if asset_name in asset_id_to_uuid:
        info = asset_id_to_uuid[asset_name]
        mapping_rows.append({
            'Asset Name' : asset_name,
            'UUID'       : info['uuid'],
            'Hierarchy'  : info['hierarchy'],
        })
        print(f"  {asset_name:6s}  →  {info['uuid']}  ({info['hierarchy']})")
    else:
        mapping_rows.append({'Asset Name': asset_name, 'UUID': None, 'Hierarchy': None})
        print(f"  {asset_name:6s}  →  [NOT FOUND in ADNOC-B.json]")

df_asset_map = pd.DataFrame(mapping_rows)
print(f"\nResolved: {df_asset_map['UUID'].notna().sum()} / {len(df_asset_map)} assets")


Unique ASSET IDs in sheet : ['1A', '1D', '1E', '1F', '1I', '1L']

  1A      →  8b999a4a-2a1e-4d6c-b76d-7a498fb5c4a3  (ADNOC\BUHASA\ADNOC-B\TRAIN_1\1A)
  1D      →  9cac06f4-131b-4af0-aea0-17b223973d12  (ADNOC\BUHASA\ADNOC-B\TRAIN_1\1D)
  1E      →  072404dc-ae93-4239-9f6b-f4624391c391  (ADNOC\BUHASA\ADNOC-B\TRAIN_1\1E)
  1F      →  ccfff937-ff8c-4f84-a31c-a41f00fa0ff8  (ADNOC\BUHASA\ADNOC-B\TRAIN_1\1F)
  1I      →  0b4d6b2d-7ed9-42ad-8ed6-6ee2bd33172c  (ADNOC\BUHASA\ADNOC-B\TRAIN_1\1I)
  1L      →  65f25414-8750-4713-b7a5-d073ee37c3d5  (ADNOC\BUHASA\ADNOC-B\TRAIN_1\1L)

Resolved: 6 / 6 assets


In [11]:
# Attach UUID back to each row in df_tags for convenience
df_tags = df_tags.merge(
    df_asset_map.rename(columns={'Asset Name': 'ASSET ID'})[['ASSET ID', 'UUID']],
    on='ASSET ID', how='left'
)

print("df_tags enriched with UUID column:")
display(df_tags[['DCS Tag Name', 'DESCRIPTION', 'ASSET ID', 'UUID']])

df_tags enriched with UUID column:


,DCS Tag Name,DESCRIPTION,ASSET ID,UUID
0,03LIC_1016,3E104 LEVEL,1E,072404dc-ae93-4239-9f6b-f4624391c391
1,03TI_1405,3E103 TUBE O/L TEMP,1E,072404dc-ae93-4239-9f6b-f4624391c391
2,03HIC_1023A,SPLIT RANGE 03PIC_1023,1E,072404dc-ae93-4239-9f6b-f4624391c391
3,03HIC_1023B,SPLITE RANGE 03PIC_1023,1E,072404dc-ae93-4239-9f6b-f4624391c391
4,03TIC_1023,3C101 OVHD TO 3V101 TEMP,1E,072404dc-ae93-4239-9f6b-f4624391c391
5,03TI_1015,C3 I/L TO 3E104 TEMP T/C,1E,072404dc-ae93-4239-9f6b-f4624391c391
6,03PDI_1020,DP ACROSS 3E104,1E,072404dc-ae93-4239-9f6b-f4624391c391
7,03TI_1404,3E102 TUBE O/L TEMP,1E,072404dc-ae93-4239-9f6b-f4624391c391
8,03TI_1005,3E102 TUBE I/L TEMP,1E,072404dc-ae93-4239-9f6b-f4624391c391
9,03PIC_1023,C3 3E104 TO 3C107 PRES,1E,072404dc-ae93-4239-9f6b-f4624391c391


## 5. Discover Available Folders in Historian_Events

The `Historian/` sub-folder holds **tag-level time-series** parquet files (one file per tag per asset).  
The `events/` sub-folder holds **alarm & event** parquet files for each asset.

We check which asset UUID folders actually exist on disk.

In [12]:
historian_folders = set(os.listdir(HISTORIAN_DIR))
events_folders    = set(os.listdir(EVENTS_DIR))

print(f"Folders found in Historian/ : {len(historian_folders)}")
print(f"Folders found in events/    : {len(events_folders)}")
print()

availability = []
for _, row in df_asset_map.iterrows():
    uuid = row['UUID']
    in_hist   = uuid in historian_folders if uuid else False
    in_events = uuid in events_folders    if uuid else False
    availability.append({
        'Asset Name'      : row['Asset Name'],
        'UUID'            : uuid,
        'In Historian/'   : in_hist,
        'In events/'      : in_events,
    })
    status = []
    if in_hist:   status.append('Historian/')
    if in_events: status.append('events/')
    status_str = ', '.join(status) if status else 'MISSING in both'
    print(f"  {row['Asset Name']:6s}  {uuid}  →  {status_str}")

df_availability = pd.DataFrame(availability)
print()
display(df_availability)

Folders found in Historian/ : 13
Folders found in events/    : 70

  1A      8b999a4a-2a1e-4d6c-b76d-7a498fb5c4a3  →  Historian/, events/
  1D      9cac06f4-131b-4af0-aea0-17b223973d12  →  Historian/, events/
  1E      072404dc-ae93-4239-9f6b-f4624391c391  →  Historian/, events/
  1F      ccfff937-ff8c-4f84-a31c-a41f00fa0ff8  →  Historian/, events/
  1I      0b4d6b2d-7ed9-42ad-8ed6-6ee2bd33172c  →  Historian/, events/
  1L      65f25414-8750-4713-b7a5-d073ee37c3d5  →  Historian/, events/



,Asset Name,UUID,In Historian/,In events/
0,1A,8b999a4a-2a1e-4d6c-b76d-7a498fb5c4a3,True,True
1,1D,9cac06f4-131b-4af0-aea0-17b223973d12,True,True
2,1E,072404dc-ae93-4239-9f6b-f4624391c391,True,True
3,1F,ccfff937-ff8c-4f84-a31c-a41f00fa0ff8,True,True
4,1I,0b4d6b2d-7ed9-42ad-8ed6-6ee2bd33172c,True,True
5,1L,65f25414-8750-4713-b7a5-d073ee37c3d5,True,True


## 6. Load Historian Time-Series Data

For each asset that has a folder in `Historian/`, list the available tag files and load  
the ones corresponding to the tags referenced in the sheet.

Each parquet file in `Historian/<UUID>/` is named `<TAG>.<PARAMETER>` (e.g. `03TIC_1023.PV`).  
Columns: `startdatetime`, `enddatetime`, `timestamp`, `value`, `S`.

In [13]:
historian_data = {}   # key: 'TAG.PARAM'  →  value: DataFrame

for _, asset_row in df_asset_map.iterrows():
    uuid       = asset_row['UUID']
    asset_name = asset_row['Asset Name']

    if not uuid or uuid not in historian_folders:
        print(f"[SKIP] {asset_name} — no Historian folder")
        continue

    asset_hist_dir = os.path.join(HISTORIAN_DIR, uuid)
    available_files = os.listdir(asset_hist_dir)

    # Tags in the sheet that belong to this asset
    asset_tags = df_tags.loc[df_tags['UUID'] == uuid, 'DCS Tag Name'].tolist()

    print(f"\n── Asset {asset_name} ({uuid}) ──")
    print(f"   Files in Historian folder : {len(available_files)}")
    print(f"   Tags in sheet for asset   : {asset_tags}")

    for tag in asset_tags:
        param = df_tags.loc[df_tags['DCS Tag Name'] == tag, 'PARAMETER'].values
        param = param[0] if len(param) > 0 else 'PV'
        filename = f"{tag}.{param}"

        filepath = os.path.join(asset_hist_dir, filename)
        if os.path.exists(filepath):
            df_hist = pd.read_parquet(filepath)
            df_hist['timestamp'] = pd.to_datetime(df_hist['timestamp'])
            historian_data[filename] = df_hist
            print(f"   ✓ Loaded  {filename:30s}  →  shape {df_hist.shape}  "
                  f"| range: {df_hist['timestamp'].min()} … {df_hist['timestamp'].max()}")
        else:
            # Try scanning available files for a match
            matches = [f for f in available_files if f.startswith(tag + '.')]
            if matches:
                for m in matches:
                    df_hist = pd.read_parquet(os.path.join(asset_hist_dir, m))
                    df_hist['timestamp'] = pd.to_datetime(df_hist['timestamp'])
                    historian_data[m] = df_hist
                    print(f"   ✓ Loaded  {m:30s}  →  shape {df_hist.shape}  "
                          f"| range: {df_hist['timestamp'].min()} … {df_hist['timestamp'].max()}")
            else:
                print(f"   ✗ Not found: {filename}")

print(f"\n{'='*60}")
print(f"Total historian series loaded: {len(historian_data)}")


── Asset 1A (8b999a4a-2a1e-4d6c-b76d-7a498fb5c4a3) ──
   Files in Historian folder : 11
   Tags in sheet for asset   : ['02FI_1000']
   ✓ Loaded  02FI_1000.PV                    →  shape (2130628, 5)  | range: 2022-01-04 01:00:00 … 2026-01-10 01:00:00

── Asset 1D (9cac06f4-131b-4af0-aea0-17b223973d12) ──
   Files in Historian folder : 15
   Tags in sheet for asset   : ['03TI_1002']
   ✓ Loaded  03TI_1002.PV                    →  shape (2131229, 5)  | range: 2022-01-04 01:00:00 … 2026-01-10 01:00:00

── Asset 1E (072404dc-ae93-4239-9f6b-f4624391c391) ──
   Files in Historian folder : 31
   Tags in sheet for asset   : ['03LIC_1016', '03TI_1405', '03HIC_1023A', '03HIC_1023B', '03TIC_1023', '03TI_1015', '03PDI_1020', '03TI_1404', '03TI_1005', '03PIC_1023', '03LI_1035', '03LIC_1034', '03TI_1024', '03PI_1409', '03TIC_1009', '03LIC_1031', '03LIC_1034']
   ✓ Loaded  03LIC_1016.PV                   →  shape (2134243, 5)  | range: 2022-01-04 01:00:00 … 2026-01-10 01:00:00
   ✓ Loaded  03TI_140

### 6a. Historian Data — Preview

In [14]:
for key, df_h in historian_data.items():
    print(f"\n── {key} ──")
    print(f"   Shape   : {df_h.shape}")
    print(f"   Columns : {df_h.columns.tolist()}")
    print(f"   Null %  : {(df_h['value'].isna().mean()*100):.2f}%  missing values")
    display(df_h.head(3))


── 02FI_1000.PV ──
   Shape   : (2130628, 5)
   Columns : ['startdatetime', 'enddatetime', 'timestamp', 'value', 'S']
   Null %  : 98.36%  missing values


,startdatetime,enddatetime,timestamp,value,S
0,2022-01-04 00:00:00,2022-01-04 01:00:00,2022-01-04 01:00:00,6.623204,NaN
1,2022-01-04 01:00:00,2022-01-04 02:00:00,2022-01-04 02:00:00,6.683493,NaN
2,2022-01-04 02:00:00,2022-01-04 03:00:00,2022-01-04 03:00:00,6.603747,NaN



── 03TI_1002.PV ──
   Shape   : (2131229, 5)
   Columns : ['startdatetime', 'enddatetime', 'timestamp', 'value', 'S']
   Null %  : 98.36%  missing values


,startdatetime,enddatetime,timestamp,value,S
0,2022-01-04 00:00:00,2022-01-04 01:00:00,2022-01-04 01:00:00,45.491426,NaN
1,2022-01-04 01:00:00,2022-01-04 02:00:00,2022-01-04 02:00:00,45.323845,NaN
2,2022-01-04 02:00:00,2022-01-04 03:00:00,2022-01-04 03:00:00,44.802269,NaN



── 03LIC_1016.PV ──
   Shape   : (2134243, 5)
   Columns : ['startdatetime', 'enddatetime', 'timestamp', 'value', 'S']
   Null %  : 98.36%  missing values


...

,startdatetime,enddatetime,timestamp,value,S
0,2022-01-04 00:00:00,2022-01-04 01:00:00,2022-01-04 01:00:00,15.676882,NaN
1,2022-01-04 01:00:00,2022-01-04 02:00:00,2022-01-04 02:00:00,15.574282,NaN
2,2022-01-04 02:00:00,2022-01-04 03:00:00,2022-01-04 03:00:00,15.420951,NaN



── 03FI_1810.PV ──
   Shape   : (2133824, 5)
   Columns : ['startdatetime', 'enddatetime', 'timestamp', 'value', 'S']
   Null %  : 98.36%  missing values


,startdatetime,enddatetime,timestamp,value,S
0,2022-01-04 00:00:00,2022-01-04 01:00:00,2022-01-04 01:00:00,134570.8968,NaN
1,2022-01-04 01:00:00,2022-01-04 02:00:00,2022-01-04 02:00:00,131715.6396,NaN
2,2022-01-04 02:00:00,2022-01-04 03:00:00,2022-01-04 03:00:00,140962.9830,NaN



── 03PIC_1013.PV ──
   Shape   : (2134280, 5)
   Columns : ['startdatetime', 'enddatetime', 'timestamp', 'value', 'S']
   Null %  : 98.36%  missing values


,startdatetime,enddatetime,timestamp,value,S
0,2022-01-04 00:00:00,2022-01-04 01:00:00,2022-01-04 01:00:00,95.155259,NaN
1,2022-01-04 01:00:00,2022-01-04 02:00:00,2022-01-04 02:00:00,106.229831,NaN
2,2022-01-04 02:00:00,2022-01-04 03:00:00,2022-01-04 03:00:00,97.623941,NaN


## 7. Load Events Data

For each asset UUID folder in `events/`, there are up to three types of parquet files:

| Suffix | Content |
|--------|---------|
| `_E`   | Process/operator change events |
| `_EA`  | Alarm events (no last/next linkage) |
| `_EAL` | Alarm events **with** linked previous/next alarm (includes duration) |

We load all three types for every relevant asset.

In [15]:
events_data = {}   # key: '<asset_name>_<type>'  →  value: DataFrame

for _, asset_row in df_asset_map.iterrows():
    uuid       = asset_row['UUID']
    asset_name = asset_row['Asset Name']

    if not uuid or uuid not in events_folders:
        print(f"[SKIP] {asset_name} — no events folder")
        continue

    asset_events_dir = os.path.join(EVENTS_DIR, uuid)
    available_files  = os.listdir(asset_events_dir)

    print(f"\n── Asset {asset_name} ({uuid}) ──")
    print(f"   Files in events folder : {available_files}")

    for suffix, label in [('_E.parquet', 'ChangeEvents'), ('_EA.parquet', 'AlarmEvents'), ('_EAL.parquet', 'AlarmEventsLinked')]:
        match = [f for f in available_files if f.endswith(suffix)]
        if match:
            fp = os.path.join(asset_events_dir, match[0])
            df_ev = pd.read_parquet(fp)
            # Parse timestamp column where available — use format='mixed' to handle
            # rows where milliseconds are present/absent in the same column
            for ts_col in ['VT_Start', 'Time', 'timestamp']:
                if ts_col in df_ev.columns:
                    df_ev[ts_col] = pd.to_datetime(df_ev[ts_col], format='mixed')
            key = f"{asset_name}_{label}"
            events_data[key] = df_ev
            print(f"   ✓ Loaded  {match[0]:50s}  →  shape {df_ev.shape}")
        else:
            print(f"   ✗ No {suffix} file found")

print(f"\n{'='*60}")
print(f"Total event tables loaded: {len(events_data)}")


── Asset 1A (8b999a4a-2a1e-4d6c-b76d-7a498fb5c4a3) ──
   Files in events folder : ['8b999a4a-2a1e-4d6c-b76d-7a498fb5c4a3.parquet', '2021011814_2026011001_EAE.parquet', '2021011814_2026011001_EA.parquet', '2021011814_2026011001_EALE.parquet', '2021011814_2026011001_EAL.parquet', '2021011814_2026011001_E.parquet']
   ✓ Loaded  2021011814_2026011001_E.parquet                     →  shape (2507555, 30)
   ✓ Loaded  2021011814_2026011001_EA.parquet                    →  shape (54844, 24)
   ✓ Loaded  2021011814_2026011001_EAL.parquet                   →  shape (201117, 29)

── Asset 1D (9cac06f4-131b-4af0-aea0-17b223973d12) ──
   Files in events folder : ['2021011814_2026011001_EAE.parquet', '9cac06f4-131b-4af0-aea0-17b223973d12.parquet', '2021011814_2026011001_EA.parquet', '2021011814_2026011001_EALE.parquet', '2021011814_2026011001_EAL.parquet', '2021011814_2026011001_E.parquet']
   ✓ Loaded  2021011814_2026011001_E.parquet                     →  shape (24300, 30)
   ✓ Loaded  2021011814

### 7a. Events Data — Preview

In [16]:
for key, df_ev in events_data.items():
    print(f"\n── {key} ──")
    print(f"   Shape   : {df_ev.shape}")
    print(f"   Columns : {df_ev.columns.tolist()}")
    display(df_ev.head(3))


── 1A_ChangeEvents ──
   Shape   : (2507555, 30)
   Columns : ['Action', 'Actor', 'AreaName', 'AlarmLimit', 'Block', 'Category', 'ConditionName', 'Description', 'EventID', 'Flags', 'LocalTime', 'LocationFullName', 'LocationTagName', 'PrevValue', 'Priority', 'ReceivedDelay', 'ServerName', 'ShelvedReason', 'Source', 'SourceParameter', 'Station', 'Time', 'TransactionID', 'Units', 'Value', 'VT_Start', 'H', 'TagID', 'AlarmStatus', 'S']


,Action,Actor,AreaName,AlarmLimit,Block,Category,ConditionName,Description,EventID,Flags,...,Station,Time,TransactionID,Units,Value,VT_Start,H,TagID,AlarmStatus,S
0,None,None,1A,NaN,None,14,None,PROCESS EVNT RCVRY 1A ...,31214430,NaN,...,None,1974-03-17 20:27:01.115608,31214430,None,None,2021-10-11 10:31:51.560800,2021_10_11_10,$CONSOLE01,None,S1-PROD-ESVT01
1,None,None,1A,NaN,None,14,None,PROCESS EVNT RCVRY 1A ...,31214424,NaN,...,None,1974-03-17 20:27:01.116026,31214424,None,None,2021-10-11 10:31:51.602600,2021_10_11_10,$CONSOLE01,None,S1-PROD-ESVT01
2,None,None,1A,NaN,None,14,None,PROCESS EVNT RCVRY 1A ...,31214423,NaN,...,None,1974-03-17 20:27:01.116028,31214423,None,None,2021-10-11 10:31:51.602800,2021_10_11_10,$CONSOLE01,None,S1-PROD-ESVT01



── 1A_AlarmEvents ──
   Shape   : (54844, 24)
   Columns : ['ServerName', 'EventID', 'IntervalIdentifier', 'Priority', 'Parameter', 'FromValue', 'ToValue', 'Value', 'Description', 'ACTION', 'Unit', 'UnitOfMeasure', 'TagName', 'Area', 'Limit', 'LocationFullName', 'LocationTagName', 'AlarmStatus', 'VT_Start', 'TagID', 'EventTypeID', 'VT_End', 'H', 'S']


,ServerName,EventID,IntervalIdentifier,Priority,Parameter,FromValue,ToValue,Value,Description,ACTION,...,Limit,LocationFullName,LocationTagName,AlarmStatus,VT_Start,TagID,EventTypeID,VT_End,H,S
0,S1-PROD-ESVT01,61059050,CHANGE,16,None,None,INACTIVE,INACTIVE,PTEXECST,None,...,NaN,/Assets/TRAIN_1/1A,1A,None,2023-05-04 13:21:48.203000,$DET0108,4,None,2023_05_04_13,S1-PROD-ESVT01
1,S1-PROD-ESVT01,61059051,None,16,None,None,None,None,PTEXECST ACTIVE INACTIVE,None,...,NaN,/Assets/TRAIN_1/1A,1A,None,2023-05-04 13:21:48.203000,$DET0108,4,None,2023_05_04_13,S1-PROD-ESVT01
2,S1-PROD-ESVT01,61059053,CHANGE,16,None,None,INACTIVE,INACTIVE,PTEXECST,None,...,NaN,/Assets/TRAIN_1/1A,1A,None,2023-05-04 13:21:55.256300,$DET0108,4,None,2023_05_04_13,S1-PROD-ESVT01



── 1A_AlarmEventsLinked ──
   Shape   : (201117, 29)
   Columns : ['ServerName', 'EventID', 'IntervalIdentifier', 'Priority', 'Parameter', 'FromValue', 'ToValue', 'Value', 'Description', 'ACTION', 'Unit', 'UnitOfMeasure', 'TagName', 'Area', 'Limit', 'LocationFullName', 'LocationTagName', 'AlarmStatus', 'VT_Start', 'TagID', 'EventTypeID', 'VT_End', 'LastEvent', 'NextEvent', 'rowid', 'Seconds', 'PrevSeconds', 'H', 'S']


...

,Action,Actor,AreaName,AlarmLimit,Block,Category,ConditionName,Description,EventID,Flags,...,Station,Time,TransactionID,Units,Value,VT_Start,H,TagID,AlarmStatus,S
0,None,None,1L,NaN,None,14,None,PROCESS EVNT RCVRY 1L ...,31214603,NaN,...,None,1974-03-17 20:27:02.044029,31214603,None,None,2021-10-11 10:33:24.402900,2021_10_11_10,$CONSOLE01,None,S1-PROD-ESVT01
1,None,None,1L,NaN,None,14,None,PROCESS EVNT RCVRY 1L ...,31214604,NaN,...,None,1974-03-17 20:27:02.044035,31214604,None,None,2021-10-11 10:33:24.403500,2021_10_11_10,$CONSOLE01,None,S1-PROD-ESVT01
2,None,None,1L,NaN,None,14,None,PROCESS EVNT RCVRY 1L ...,31214605,NaN,...,None,1974-03-17 20:27:02.045100,31214605,None,None,2021-10-11 10:33:24.510000,2021_10_11_10,$CONSOLE01,None,S1-PROD-ESVT01



── 1L_AlarmEvents ──
   Shape   : (86427, 24)
   Columns : ['ServerName', 'EventID', 'IntervalIdentifier', 'Priority', 'Parameter', 'FromValue', 'ToValue', 'Value', 'Description', 'ACTION', 'Unit', 'UnitOfMeasure', 'TagName', 'Area', 'Limit', 'LocationFullName', 'LocationTagName', 'AlarmStatus', 'VT_Start', 'TagID', 'EventTypeID', 'VT_End', 'H', 'S']


,ServerName,EventID,IntervalIdentifier,Priority,Parameter,FromValue,ToValue,Value,Description,ACTION,...,Limit,LocationFullName,LocationTagName,AlarmStatus,VT_Start,TagID,EventTypeID,VT_End,H,S
0,S1-PROD-ESVT01,46282212,MESSAGE,16,None,None,None,None,20May22 17:32:26 * Completed AM Schedule Data ...,ACK,...,NaN,/Assets/TRAIN_1/1L,1L,None,2022-05-20 19:04:29.145400,$Prsts30,4,None,2022_05_20_19,S1-PROD-ESVT01
1,S1-PROD-ESVT01,31214606,None,16,None,None,None,None,UT ALM RECOV ...,None,...,NaN,/Assets/TRAIN_1/1L,1L,None,2021-10-11 10:33:24.509500,$UNITOPS,4,None,2021_10_11_10,S1-PROD-ESVT01
2,S1-PROD-ESVT01,31214607,None,16,None,None,None,None,UT RECOV CMP ...,None,...,NaN,/Assets/TRAIN_1/1L,1L,None,2021-10-11 10:33:24.509500,$UNITOPS,4,None,2021_10_11_10,S1-PROD-ESVT01



── 1L_AlarmEventsLinked ──
   Shape   : (49369, 29)
   Columns : ['ServerName', 'EventID', 'IntervalIdentifier', 'Priority', 'Parameter', 'FromValue', 'ToValue', 'Value', 'Description', 'ACTION', 'Unit', 'UnitOfMeasure', 'TagName', 'Area', 'Limit', 'LocationFullName', 'LocationTagName', 'AlarmStatus', 'VT_Start', 'TagID', 'EventTypeID', 'VT_End', 'LastEvent', 'NextEvent', 'rowid', 'Seconds', 'PrevSeconds', 'H', 'S']


,ServerName,EventID,IntervalIdentifier,Priority,Parameter,FromValue,ToValue,Value,Description,ACTION,...,TagID,EventTypeID,VT_End,LastEvent,NextEvent,rowid,Seconds,PrevSeconds,H,S
0,S1-PROD-ESVT01,32781538,CMDDIS,16,None,None,BAD,BAD,3K101 ESD VENT VALV,None,...,03EDPV_1153,1,None,0,2,1,1452,6,2021_12_04_19,S1-PROD-ESVT01
1,S1-PROD-ESVT01,32783017,CMDDIS,16,None,None,CLOSE,CLOSE,3K101 ESD VENT VALV,OK,...,03EDPV_1153,2,None,1,1,2,2065,1452,2021_12_04_19,S1-PROD-ESVT01
2,S1-PROD-ESVT01,32784906,CMDDIS,16,None,None,CLOSE,CLOSE,3K101 ESD VENT VALV,None,...,03EDPV_1153,1,None,2,2,3,120,2065,2021_12_04_20,S1-PROD-ESVT01


## 8. Summary

In [17]:
print(f"{'='*60}")
print(f"  SUMMARY for target tag: {TARGET_TAG}")
print(f"{'='*60}")
print(f"  Related tags in sheet         : {len(df_tags)}")
print(f"  Unique assets referenced      : {len(df_asset_map)}")
print(f"  Assets found in Historian/    : {df_availability['In Historian/'].sum()}")
print(f"  Assets found in events/       : {df_availability['In events/'].sum()}")
print(f"  Historian series loaded       : {len(historian_data)}")
print(f"  Event tables loaded           : {len(events_data)}")
print(f"{'='*60}")
print()
print("Historian series:")
for k, v in historian_data.items():
    print(f"  {k:35s}  {v.shape[0]:>8,} rows")
print()
print("Event tables:")
for k, v in events_data.items():
    print(f"  {k:45s}  {v.shape[0]:>8,} rows")

  SUMMARY for target tag: 03PIC1023
  Related tags in sheet         : 34
  Unique assets referenced      : 6
  Assets found in Historian/    : 6
  Assets found in events/       : 6
  Historian series loaded       : 32
  Event tables loaded           : 18

Historian series:
  02FI_1000.PV                         2,130,628 rows
  03TI_1002.PV                         2,131,229 rows
  03LIC_1016.PV                        2,134,243 rows
  03TI_1405.PV                         2,134,310 rows
  03HIC_1023A.OP                       2,134,280 rows
  03HIC_1023B.OP                       2,134,284 rows
  03TIC_1023.PV                        2,134,314 rows
  03TI_1015.PV                         2,134,314 rows
  03PDI_1020.PV                        2,134,283 rows
  03TI_1404.PV                         2,134,310 rows
  03TI_1005.PV                         2,134,313 rows
  03PIC_1023.PV                        2,134,283 rows
  03LI_1035.PV                         2,133,855 rows
  03LIC_1034.PV         

## 9. Combine All Events and Save

Concatenate all event tables (`_E`, `_EA`, `_EAL`) for all assets into a single DataFrame,
add a `source_table` column to track the origin, sort by `VT_Start`, and save to
`DATA/combined_events/<TARGET_TAG>_combined_events.parquet`.

In [18]:
import os

# ── Output directory ────────────────────────────────────────────────────────────
OUTPUT_DIR = os.path.join(DATA_DIR, 'combined_events')
os.makedirs(OUTPUT_DIR, exist_ok=True)
print(f"Output directory : {OUTPUT_DIR}")
print()

# ── Concatenate all event DataFrames ────────────────────────────────────────────
print(f"Concatenating {len(events_data)} event tables ...\n")

parts = []
for table_key, df_part in events_data.items():
    df_part = df_part.copy()
    df_part['source_table'] = table_key          # track origin: e.g. '1E_AlarmEvents'
    parts.append(df_part)
    print(f"  + {table_key:45s}  {len(df_part):>9,} rows  |  "
          f"VT_Start range: {df_part['VT_Start'].min()}  →  {df_part['VT_Start'].max()}")

df_combined = pd.concat(parts, ignore_index=True, sort=False)

print(f"\nRaw combined shape : {df_combined.shape}")
print(f"Columns            : {df_combined.columns.tolist()}")

# ── Apply timezone offset ─────────────────────────────────────────────────────────
# Historian events are stored in UTC+5:30 (IST); plant PV data is in UTC+4 (UAE local).
# Subtract 1.5 hours to align event timestamps with the PV time series.
TIME_OFFSET = pd.Timedelta(hours=1.5)
print(f"\nApplying timezone offset: VT_Start -= {TIME_OFFSET} (UTC+5:30 → UAE local UTC+4)")
print(f"  VT_Start before: {df_combined['VT_Start'].min()}  →  {df_combined['VT_Start'].max()}")
df_combined['VT_Start'] = df_combined['VT_Start'] - TIME_OFFSET
print(f"  VT_Start after : {df_combined['VT_Start'].min()}  →  {df_combined['VT_Start'].max()}")

# ── Sort by VT_Start ─────────────────────────────────────────────────────────────
print(f"\nSorting by VT_Start ...")
df_combined.sort_values('VT_Start', inplace=True, ignore_index=True)

print(f"Sorted combined shape : {df_combined.shape}")
print(f"VT_Start range        : {df_combined['VT_Start'].min()}  →  {df_combined['VT_Start'].max()}")
print(f"Null VT_Start rows    : {df_combined['VT_Start'].isna().sum():,}")

# ── Save ─────────────────────────────────────────────────────────────────────────
output_path = os.path.join(OUTPUT_DIR, f"{TARGET_TAG}_combined_events.parquet")
df_combined.to_parquet(output_path, index=False)
file_size_mb = os.path.getsize(output_path) / (1024 ** 2)

print(f"\n{'='*60}")
print(f"  Saved to : {output_path}")
print(f"  Rows     : {len(df_combined):,}")
print(f"  Columns  : {len(df_combined.columns)}")
print(f"  File size: {file_size_mb:.1f} MB")
print(f"{'='*60}")
print(f"\nsource_table breakdown:")
print(df_combined['source_table'].value_counts().to_string())

Output directory : /home/h604827/ControlActions/DATA/combined_events

Concatenating 18 event tables ...

  + 1A_ChangeEvents                                2,507,555 rows  |  VT_Start range: 2021-02-01 17:19:41.804300  →  2025-06-28 00:22:59.137300
  + 1A_AlarmEvents                                    54,844 rows  |  VT_Start range: 2021-10-02 14:30:37.668800  →  2025-06-27 18:53:34.052000
  + 1A_AlarmEventsLinked                             201,117 rows  |  VT_Start range: 2021-10-09 09:58:38.952700  →  2025-06-03 10:22:26.768300
  + 1D_ChangeEvents                                   24,300 rows  |  VT_Start range: 2021-10-02 10:00:03.156200  →  2025-06-28 05:02:20.589100
  + 1D_AlarmEvents                                     7,965 rows  |  VT_Start range: 2021-10-06 07:08:00.690700  →  2025-06-27 18:10:42.546300
  + 1D_AlarmEventsLinked                               2,170 rows  |  VT_Start range: 2021-10-06 07:07:52.002300  →  2025-06-27 18:34:03.234400
  + 1E_ChangeEvents            

## 10. Batch Processing — All Tags

Processes **all alarm-tag sheets** from the Excel workbook (from the first tag sheet through to `03PI_1655_PVHI`).

For each sheet:
- The alarm type (`PVLO`, `PVHI`, `PVLO_PVHI`, `AOA`) is inferred from the sheet name.
- Sheets with no valid `ASSET ID` values are skipped automatically.
- Generic non-alarm sheets (`Sheet`, `Sheet1`, `Sheet2`) are skipped.
- Output files are named `{TAG}_{ALARM_TYPE}_combined_events.parquet` and saved to `DATA/combined_events/`.
- Files that already exist are skipped (set `SKIP_EXISTING = False` to re-process them).


In [19]:
import re

# ── Batch config ────────────────────────────────────────────────────────────────
STOP_SHEET    = '03PI_1655_PVHI'    # last sheet to process (inclusive)
SKIP_EXISTING = True                 # set False to re-process already-saved files

# Generic sheets that do not correspond to alarm tags
SKIP_SHEETS = {'Tag List', 'Sheet', 'Sheet1', 'Sheet2'}

# Asset IDs that are invalid / placeholder values
INVALID_ASSET_IDS = {'nan', 'NO', '', 'none'}


def parse_sheet_name(sheet_name):
    """
    Infer (tag, alarm_type) from a sheet name.

    Handles both space- and underscore-separated suffixes, e.g.:
      '03TIC_1023 PVLO_PVHI'  →  ('03TIC_1023', 'PVLO_PVHI')
      '03TIC_1009_PVLO'       →  ('03TIC_1009', 'PVLO')
      '03LIC1608_PVLO'        →  ('03LIC1608',  'PVLO')
      '03LIC_1094'            →  ('03LIC_1094',  None)
    """
    # Check longer alarm types first to avoid PVLO_PVHI matching as PVLO
    for alarm_type in ('PVLO_PVHI', 'PVLO', 'PVHI', 'AOA'):
        for sep in (' ', '_'):
            suffix = sep + alarm_type
            if sheet_name.endswith(suffix):
                return sheet_name[: -len(suffix)].strip(), alarm_type
    return sheet_name.strip(), None


# ── Discover sheets to process ──────────────────────────────────────────────────
_wb = openpyxl.load_workbook(EXCEL_FILE, read_only=True, data_only=True)
_all_sheets = _wb.sheetnames
_wb.close()

_stop_idx = _all_sheets.index(STOP_SHEET) if STOP_SHEET in _all_sheets else len(_all_sheets) - 1
# Skip the very first sheet ('Tag List') and stop at STOP_SHEET (inclusive)
_candidate_sheets = [s for s in _all_sheets[1:_stop_idx + 1] if s not in SKIP_SHEETS]

print(f"Candidate sheets to process : {len(_candidate_sheets)}")
for _s in _candidate_sheets:
    _t, _a = parse_sheet_name(_s)
    _alarm_suffix = f'_{_a}' if _a else ''
    _fname = f"{_t}{_alarm_suffix}_combined_events.parquet"
    _exists = os.path.exists(os.path.join(DATA_DIR, 'combined_events', _fname))
    print(f"  {'[EXISTS]' if _exists else '        ':10s}  {_s!r:38s}  →  {_fname}")


Candidate sheets to process : 19
  [EXISTS]    '03TIC_1023 PVLO_PVHI'                  →  03TIC_1023_PVLO_PVHI_combined_events.parquet
  [EXISTS]    '03TIC_1635 PVLO'                       →  03TIC_1635_PVLO_combined_events.parquet
  [EXISTS]    '03TIC_1009_PVLO'                       →  03TIC_1009_PVLO_combined_events.parquet
  [EXISTS]    '03TIC_1009_PVHI'                       →  03TIC_1009_PVHI_combined_events.parquet
  [EXISTS]    '03LIC_1619 PVLO_PVHI'                  →  03LIC_1619_PVLO_PVHI_combined_events.parquet
  [EXISTS]    '03TIC_1635_PVHI'                       →  03TIC_1635_PVHI_combined_events.parquet
  [EXISTS]    '03FIC_1668 PVHI'                       →  03FIC_1668_PVHI_combined_events.parquet
  [EXISTS]    '03PIC_1104 PVHI'                       →  03PIC_1104_PVHI_combined_events.parquet
  [EXISTS]    '03PIC1023 PVLO_PVHI'                   →  03PIC1023_PVLO_PVHI_combined_events.parquet
  [EXISTS]    '03TIC_1145 PVLO'                       →  03TIC_1145_PVLO_combine

In [20]:
# import traceback

# # ── Load ADNOC-B.json once (reuse asset_id_to_uuid if already loaded) ───────────
# with open(ADNOC_JSON, 'r') as f:
#     _asset_nodes = json.load(f)
# _asset_id_to_uuid = {n['Name']: {'uuid': n['Id'], 'hierarchy': n['Hierarchy']} for n in _asset_nodes}

# _hist_folders   = set(os.listdir(HISTORIAN_DIR))
# _events_folders = set(os.listdir(EVENTS_DIR))

# OUTPUT_DIR   = os.path.join(DATA_DIR, 'combined_events')
# os.makedirs(OUTPUT_DIR, exist_ok=True)

# TIME_OFFSET  = pd.Timedelta(hours=1.5)

# _KEEP_COLS = ['Potential Input parameter', 'DCS Tag Name', 'DESCRIPTION',
#               'PARAMETER', 'IS DATA AVAILABLE?', 'ASSET ID']

# batch_summary = []

# for _sheet_name in _candidate_sheets:
#     _tag, _alarm_type = parse_sheet_name(_sheet_name)
#     _alarm_suffix  = f'_{_alarm_type}' if _alarm_type else ''
#     _out_filename  = f"{_tag}{_alarm_suffix}_combined_events.parquet"
#     _out_path      = os.path.join(OUTPUT_DIR, _out_filename)

#     # ── Skip if output already exists ─────────────────────────────────────────────
#     if SKIP_EXISTING and os.path.exists(_out_path):
#         _mb = os.path.getsize(_out_path) / (1024 ** 2)
#         print(f"[SKIP  ] {_sheet_name!r:38s}  →  already exists ({_mb:.1f} MB)")
#         batch_summary.append({'Sheet': _sheet_name, 'Tag': _tag, 'AlarmType': _alarm_type,
#                                'Status': 'SKIPPED (exists)', 'Rows': None, 'File': _out_filename})
#         continue

#     print(f"\n{'='*72}")
#     print(f"  Sheet : {_sheet_name!r}")
#     print(f"  Tag   : {_tag}   AlarmType: {_alarm_type}   Output: {_out_filename}")
#     print(f"{'='*72}")

#     try:
#         # ── Read sheet ─────────────────────────────────────────────────────────────
#         _df_sheet = pd.read_excel(EXCEL_FILE, sheet_name=_sheet_name, header=EXCEL_HEADER_ROW)

#         if 'ASSET ID' not in _df_sheet.columns or 'DCS Tag Name' not in _df_sheet.columns:
#             print(f"  [SKIP] Required columns (ASSET ID / DCS Tag Name) not found")
#             batch_summary.append({'Sheet': _sheet_name, 'Tag': _tag, 'AlarmType': _alarm_type,
#                                    'Status': 'SKIP (missing cols)', 'Rows': None, 'File': None})
#             continue

#         _avail_cols = [c for c in _KEEP_COLS if c in _df_sheet.columns]
#         _df_tags_b = (_df_sheet[_avail_cols]
#                       .dropna(subset=['DCS Tag Name'])
#                       .copy())
#         _df_tags_b['DCS Tag Name'] = _df_tags_b['DCS Tag Name'].astype(str).str.strip()
#         _df_tags_b['ASSET ID']     = _df_tags_b['ASSET ID'].astype(str).str.strip()
#         # Filter out invalid / blank asset IDs
#         _df_tags_b = _df_tags_b[~_df_tags_b['ASSET ID'].str.lower().isin(INVALID_ASSET_IDS)]
#         _df_tags_b = _df_tags_b.reset_index(drop=True)

#         if _df_tags_b.empty:
#             print(f"  [SKIP] No valid ASSET IDs found in sheet")
#             batch_summary.append({'Sheet': _sheet_name, 'Tag': _tag, 'AlarmType': _alarm_type,
#                                    'Status': 'SKIP (no assets)', 'Rows': None, 'File': None})
#             continue

#         _unique_assets = sorted(_df_tags_b['ASSET ID'].unique())
#         print(f"  Asset IDs : {_unique_assets}")

#         # ── Map asset names → UUIDs ────────────────────────────────────────────────
#         _asset_map_rows = []
#         for _an in _unique_assets:
#             if _an in _asset_id_to_uuid:
#                 _info = _asset_id_to_uuid[_an]
#                 _asset_map_rows.append({'Asset Name': _an, 'UUID': _info['uuid'],
#                                         'Hierarchy': _info['hierarchy']})
#             else:
#                 print(f"    [WARN] Asset '{_an}' not found in ADNOC-B.json — skipping")
#                 _asset_map_rows.append({'Asset Name': _an, 'UUID': None, 'Hierarchy': None})
#         _df_am = pd.DataFrame(_asset_map_rows)

#         # Merge UUID back to df_tags_b
#         _df_tags_b = _df_tags_b.merge(
#             _df_am.rename(columns={'Asset Name': 'ASSET ID'})[['ASSET ID', 'UUID']],
#             on='ASSET ID', how='left')

#         # ── Load events data ───────────────────────────────────────────────────────
#         _events_data_b = {}
#         for _, _arow in _df_am.iterrows():
#             _uuid  = _arow['UUID']
#             _aname = _arow['Asset Name']
#             if not _uuid or _uuid not in _events_folders:
#                 continue
#             _aev_dir = os.path.join(EVENTS_DIR, _uuid)
#             _aev_files = os.listdir(_aev_dir)
#             for _suf, _lbl in [('_E.parquet',   'ChangeEvents'),
#                                 ('_EA.parquet',  'AlarmEvents'),
#                                 ('_EAL.parquet', 'AlarmEventsLinked')]:
#                 _m = [f for f in _aev_files if f.endswith(_suf)]
#                 if _m:
#                     _df_ev = pd.read_parquet(os.path.join(_aev_dir, _m[0]))
#                     for _ts_col in ['VT_Start', 'Time', 'timestamp']:
#                         if _ts_col in _df_ev.columns:
#                             _df_ev[_ts_col] = pd.to_datetime(_df_ev[_ts_col], format='mixed')
#                     _events_data_b[f"{_aname}_{_lbl}"] = _df_ev

#         print(f"  Event tables loaded : {len(_events_data_b)}")

#         if not _events_data_b:
#             print(f"  [SKIP] No event parquet files found for any asset")
#             batch_summary.append({'Sheet': _sheet_name, 'Tag': _tag, 'AlarmType': _alarm_type,
#                                    'Status': 'SKIP (no events)', 'Rows': None, 'File': None})
#             continue

#         # ── Concatenate ────────────────────────────────────────────────────────────
#         _parts = []
#         for _tkey, _dfp in _events_data_b.items():
#             _dfp = _dfp.copy()
#             _dfp['source_table'] = _tkey
#             _parts.append(_dfp)

#         _df_combined = pd.concat(_parts, ignore_index=True, sort=False)

#         # Apply timezone offset (UTC+5:30 → UAE local UTC+4, i.e. subtract 1.5 h)
#         _df_combined['VT_Start'] = _df_combined['VT_Start'] - TIME_OFFSET
#         _df_combined.sort_values('VT_Start', inplace=True, ignore_index=True)

#         # ── Save ───────────────────────────────────────────────────────────────────
#         _df_combined.to_parquet(_out_path, index=False)
#         _mb = os.path.getsize(_out_path) / (1024 ** 2)
#         print(f"  ✓ Saved  {_out_filename}  ({len(_df_combined):,} rows, {_mb:.1f} MB)")
#         batch_summary.append({'Sheet': _sheet_name, 'Tag': _tag, 'AlarmType': _alarm_type,
#                                'Status': 'OK', 'Rows': len(_df_combined), 'File': _out_filename})

#     except Exception as _ex:
#         print(f"  [ERROR] {_ex}")
#         traceback.print_exc()
#         batch_summary.append({'Sheet': _sheet_name, 'Tag': _tag, 'AlarmType': _alarm_type,
#                                'Status': f'ERROR: {_ex}', 'Rows': None, 'File': None})

# # ── Final summary ──────────────────────────────────────────────────────────────
# print(f"\n{'='*72}")
# print(f"  BATCH COMPLETE  —  {len(batch_summary)} sheets processed")
# print(f"{'='*72}")
# df_batch_summary = pd.DataFrame(batch_summary)
# display(df_batch_summary)
